# PEA

PEA(PyPSA-EUR analysis) is a simple and fast tool for analyzing `network.nc`.

## Get Start

### 1. Create a pea instance

In [ ]:
import pypsa
from pea import Pea
from pea import carriers as cs

n = pypsa.Network("../../results/smart/current/networks/base_s_27_3H_3H_2045.nc")
pea = Pea(
    n,
    config={
        "zone": "DE",  # filter DE bus
    },
)

### 2. view static data

In [ ]:
pea.get(cs.onwind).df

Pass `cs.onwind` preset to restrict the carrier to `onwind`, use `df()` to view static data. Finally, display the onwind data for Germany. Just like `n.generators`

You can see all carriers preset in `/research/notebook/pea/carrier.py`. eg: `cs.offwind`, `cs.solar`, `cs.phs` and others. 

Some carrier presets contain multiple carriers. For example, `cs.offwind` includes `offwind-ac`, `offwind-dc`, `offwind-flat`:

In [ ]:
pea.get(cs.offwind).df

### 3. View time series data

In [ ]:
pea.get(cs.onwind).t("p")

 use `t("p")` to view time series data, just like `n.generators_t.p`. You can also view other time series data, eg:  `t('status')`,`t('p1')` (for link or line), `t('e')`(for store).

### 4. calculate metrics 

In [ ]:
onwind = pea.get(cs.onwind)
print("Total CAPEX: ", onwind.capex())
print("Total optimized power: ", onwind.p_nom_opt())
print("Total total electricity generation", onwind.energy())

For the Link and line components, when calculating energy, it is necessary to pass in `p0`, `p1`, `p2` to specify which node's values to record.

In [ ]:
gasPower = pea.get(cs.gasPower)  # gasPower is OCGT and  OCCT
print("Total CAPEX: ", onwind.capex())
print("Total optimized power: ", gasPower.p_nom_opt())
print("Total total electricity generation", gasPower.energy("p1"))

### 5. calculate import or export

In [ ]:
pea.get(cs.power, type="export").exportP()

Displayed the power time series data of link and line exports from Germany. Reverse lines have been considered, and the values are taken as absolute values. 

`cs.power` refers to AC and DC lines. Use `cs.h2Pipeline` to calculate the import and export volumes of hydrogen, and use `cs.gasPipeline` to calculate the import and export volumes of gas.

An import time graph can be plotted.

In [ ]:
pea.get(cs.power, type="export").exportP().sum(axis=1).plot(
    figsize=(15, 5), color=cs.colors[0]
)

Calculate total imported electricity:

In [ ]:
pea.get(cs.power, type="import").importP().sum(axis=1).sum()

### 6. make Beautiful Diagram

This is an example illustrating the installed capacity of different devices. Except, use readable names and change colors.

In [ ]:
import pandas as pd

s = pd.Series()

# Use carrier presets to calculate the optimized capacity for different types of carriers.
for carrier in [
    "onwind",
    "offwind",
    "solar",
    "solarRooftop",
    "batteryDischarger",
    "gasH2Power",
    "biomassCHP",
    "v2g",
]:
    s[carrier] = pea.get(getattr(cs, carrier)).df["p_nom_opt"].sum()

# Use readable name.
s = s.rename(index=cs.mapName)

# WM -> TW
s = s / 1e3

# Use color presets.
colorMap = [cs.mapNameColor[i] for i in s.index]

# plot bar
s.plot(kind="bar", color=colorMap)